# 00 — Environment Check

Run this notebook before starting any other phase.
Every cell should complete without errors.

## 1. Python and package versions

In [ ]:
import sys
print(f'Python: {sys.version}')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA device: {torch.cuda.get_device_name(0)}')
    print(f'CUDA version: {torch.version.cuda}')

import transformers
print(f'Transformers: {transformers.__version__}')

import datasets
print(f'Datasets: {datasets.__version__}')

import sklearn
print(f'Scikit-learn: {sklearn.__version__}')

import numpy as np
print(f'NumPy: {np.__version__}')

import pandas as pd
print(f'Pandas: {pd.__version__}')

import omegaconf
print(f'OmegaConf: {omegaconf.__version__}')

## 2. Reproducibility utility

In [ ]:
import sys; sys.path.insert(0, '..')
from src.utils.reproducibility import set_all_seeds
set_all_seeds(42)
print('Seeds set.')

## 3. Config loader

In [ ]:
from src.utils.config import load_config
cfg = load_config('../configs/base.yaml', '../configs/data.yaml')
print(f'Seed from config: {cfg.seed}')
print(f'Primary dataset: {cfg.primary_dataset}')
print('Config loader OK.')

## 4. Qwen2.5-1.5B-Instruct — tokenizer load check

In [ ]:
from transformers import AutoTokenizer

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

test_text = 'The mitochondria is the powerhouse of the cell.'
tokens = tokenizer(test_text, return_tensors='pt')
print(f'Tokenizer vocab size: {tokenizer.vocab_size}')
print(f'Test token count: {tokens.input_ids.shape[1]}')
print('Tokenizer load OK.')

## 5. Qwen2.5-1.5B-Instruct — model load check (verifies architecture)

In [ ]:
import torch
from transformers import AutoModelForCausalLM

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Loading model on: {device}')

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map=device,
    trust_remote_code=True,
)
model.eval()

n_layers = len(model.model.layers)
hidden_dim = model.config.hidden_size
print(f'Number of transformer layers: {n_layers}')   # expect 28
print(f'Hidden dim: {hidden_dim}')                   # expect 1536
assert n_layers == 28, f'Expected 28 layers, got {n_layers}'
assert hidden_dim == 1536, f'Expected hidden_dim 1536, got {hidden_dim}'
print('Model architecture verified OK.')

## 6. Forward hook smoke test — verify layer output tuple structure

In [ ]:
captured = {}

def test_hook(module, input, output):
    # output[0] should be the residual stream: (batch, seq_len, hidden_dim)
    captured['shape'] = output[0].shape
    captured['dtype'] = output[0].dtype

handle = model.model.layers[0].register_forward_hook(test_hook)

with torch.no_grad():
    _ = model(**tokens.to(device))

handle.remove()

print(f'Layer 0 output[0] shape: {captured["shape"]}')
print(f'Layer 0 output[0] dtype: {captured["dtype"]}')
assert captured['shape'][-1] == 1536, 'Hidden dim mismatch in hook output'
print('Forward hook smoke test OK.')

## 7. BEIR package check

In [ ]:
from beir.datasets.data_loader import GenericDataLoader
print('BEIR import OK.')

## 8. rank_bm25 check

In [ ]:
from rank_bm25 import BM25Okapi
corpus = [['hello', 'world'], ['foo', 'bar', 'baz']]
bm25 = BM25Okapi(corpus)
scores = bm25.get_scores(['hello'])
print(f'BM25 test scores: {scores}')
print('rank_bm25 OK.')

## All checks passed

If every cell above completed without errors, the environment is ready.
Proceed to `01_data_exploration.ipynb`.